<figure>
  <img src="https://raw.githubusercontent.com/shadowkshs/DimABSA2026/refs/heads/main/banner.png" width="100%">
</figure>

In [ ]:
import json
from typing import List, Dict
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel

from scipy.stats import pearsonr
from tqdm import tqdm
import math
import re
import requests

import src.utils
from src.models.svr import run_svr_baseline

### Step 1: Load the JSONL datasets


In [ ]:
# Configuration
subtask = "subtask_1"
task = "task1"
lang = "eng"
domain = "laptop"

train_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_train_alltasks.jsonl"
predict_url = f"https://raw.githubusercontent.com/DimABSA/DimABSA2026/refs/heads/main/task-dataset/track_a/{subtask}/{lang}/{lang}_{domain}_dev_{task}.jsonl"

models_dict = [
    {
        "name": "SVR_Baseline",
        "type": "sklearn",
        "max_features": 5000
    },
    {
        "name": "microsoft/deberta-v3-base", # microsoft/mdeberta-v3-base pour la version multilingue
        "type": "transformer",
        "lr": 1e-5,
        "epochs": 8,
        "batch_size": 32,
        "dropout": 0.1
    },
    {
        "name": "bert-base-multilingual-cased",
        "type": "transformer",
        "lr": 1e-5,
        "epochs": 8,
        "batch_size": 32,
        "dropout": 0.1
    },
    {
        "name": "roberta-base",
        "type": "transformer",
        "lr": 1e-6,
        "epochs": 8,
        "batch_size": 32,
        "dropout": 0.1
    }
]

train_raw = load_jsonl_url(train_url)
predict_raw = load_jsonl_url(predict_url)

train_df = jsonl_to_df(train_raw)
predict_df = jsonl_to_df(predict_raw)

# split 10% for dev
train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42)

### Display the dataframe

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"### {subtask}_{lang}_{domain} train_df"))
display(train_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} dev_df"))
display(dev_df.head())

display(Markdown(f"### {subtask}_{lang}_{domain} predict_df"))
display(predict_df.head())

### Définition des classes et des fonctions


In [ ]:
class VADataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.sentences = dataframe["Text"].tolist()
        self.aspects = dataframe["Aspect"].tolist()
        self.labels = dataframe[["Valence", "Arousal"]].values.astype(float)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        text = f"{self.aspects[idx]}: {self.sentences[idx]}"
        encoded = self.tokenizer(
            text, truncation=True, padding="max_length", max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

class TransformerVARegressor(nn.Module):
    def __init__(self, current_model_name, dropout=0.1):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(current_model_name)
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Linear(self.backbone.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0]
        x = self.dropout(cls_output)
        return self.reg_head(x)

    def train_epoch(self, dataloader, optimizer, loss_fn, device):
        self.train()
        total_loss = 0
        for batch in tqdm(dataloader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = self(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        return total_loss / len(dataloader)

    def eval_epoch(self, dataloader, loss_fn, device):
        self.eval()
        total_loss = 0
        with torch.no_grad():
            for batch in dataloader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = self(input_ids, attention_mask)
                loss = loss_fn(outputs, labels)
                total_loss += loss.item()
        return total_loss / len(dataloader)

In [ ]:
def get_prd(model,dataloder, type ="dev"):
    if type == "dev":
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in dataloder:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].cpu().numpy()
                outputs = model(input_ids, attention_mask).cpu().numpy()
                all_preds.append(outputs)
                all_labels.append(labels)
        preds = np.vstack(all_preds)
        lables = np.vstack(all_labels)

        pred_v = preds[:,0]
        pred_a = preds[:,1]

        gold_v = lables[:,0]
        gold_a = lables[:,1]

        return pred_v, pred_a, gold_v, gold_a

    elif type == "pred":
        all_preds = []
        with torch.no_grad():
            for batch in dataloder:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                outputs = model(input_ids, attention_mask).cpu().numpy()
                all_preds.append(outputs)
        preds = np.vstack(all_preds)

        pred_v = preds[:, 0]
        pred_a = preds[:, 1]

        return pred_v, pred_a

def evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v, is_norm=False):
    if not (all(1 <= x <= 9 for x in pred_v) and all(1 <= x <= 9 for x in pred_a)):
        print(f"Warning: Some predicted values are out of the numerical range.")

    # Calcul du PCC (Pearson Correlation Coefficient)
    pcc_v = pearsonr(pred_v, gold_v)[0]
    pcc_a = pearsonr(pred_a, gold_a)[0]

    # Calcul du RMSE séparé pour la Valence et l'Arousal avec Numpy
    rmse_v = np.sqrt(np.mean((gold_v - pred_v)**2))
    rmse_a = np.sqrt(np.mean((gold_a - pred_a)**2))

    # Calcul du RMSE global
    gold_va = np.concatenate((gold_v, gold_a))
    pred_va = np.concatenate((pred_v, pred_a))
    rmse_va_global = np.sqrt(np.mean((gold_va - pred_va)**2))

    # Appliquation de la logique de normalisation si is_norm est True
    if is_norm:
        rmse_v = rmse_v / math.sqrt(128)
        rmse_a = rmse_a / math.sqrt(128)
        rmse_va_global = rmse_va_global / math.sqrt(128)

    return {
        'PCC_V': pcc_v,
        'PCC_A': pcc_a,
        'RMSE_V': rmse_v,
        'RMSE_A': rmse_a,
        'RMSE_VA': rmse_va_global
    }

### Entraînement des modèles

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_results = {} # Pour stocker les scores finaux

for config in models_dict:
    current_model = config["name"]
    model_type = config["type"]

    print(f"\n{'='*50}")
    print(f"ENTRAÎNEMENT DU MODÈLE : {current_model}")
    print(f"{'='*50}")

    # Pipline Deep learning
    if model_type == "transformer":

        current_lr = config["lr"]
        current_epochs = config["epochs"]
        current_batch_size = config["batch_size"]
        current_dropout = config["dropout"]

        print(f"Paramètres : LR={current_lr}, Epochs={current_epochs}, Batch={current_batch_size}, Dropout={current_dropout}")

        # Création du Tokenizer et des DataLoaders
        tokenizer = AutoTokenizer.from_pretrained(current_model)
        train_dataset = VADataset(train_df, tokenizer)
        dev_dataset = VADataset(dev_df, tokenizer)

        train_loader = DataLoader(train_dataset, batch_size=current_batch_size, shuffle=True)
        dev_loader = DataLoader(dev_dataset, batch_size=current_batch_size, shuffle=False)

        # Initialisation du modèle
        model = TransformerVARegressor(current_model_name=current_model, dropout=current_dropout).to(device).float()
        optimizer = torch.optim.AdamW(model.parameters(), lr=current_lr)
        loss_fn = nn.MSELoss()

        # Entraînement du modèle
        for epoch in range(current_epochs):
            train_loss = model.train_epoch(train_loader, optimizer, loss_fn, device)
            val_loss = model.eval_epoch(dev_loader, loss_fn, device)
            print(f"Epoch {epoch+1}/{current_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        # Évaluation du modèle sur le Dev Set
        pred_v, pred_a, gold_v, gold_a = get_prd(model, dev_loader, type="dev")
        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_model] = eval_score

    # Pipline Machine Learning
    elif model_type == "sklearn":

        max_features = config["max_features"]

        pred_v, pred_a, gold_v, gold_a = run_svr_baseline(train_df, dev_df, max_features=max_features)

        eval_score = evaluate_predictions_task1(pred_a, pred_v, gold_a, gold_v)
        model_results[current_model] = eval_score

### Affichage des résultats

In [ ]:
print("\n RÉCAPITULATIF DES RÉSULTATS")
for mod, scores in model_results.items():
    print(f"- {mod} : PCC_V = {scores['PCC_V']:.4f} | PCC_A = {scores['PCC_A']:.4f} | RMSE_V = {scores['RMSE_V']:.4f} | RMSE_A = {scores['RMSE_A']:.4f}")